## Using sqlitesearch vector search in RAG

In [2]:
# make sure .env file is loaded
from dotenv import load_dotenv
print(load_dotenv())

# check whether the OpenAI client works
from openai import OpenAI
openai_client = OpenAI()

True


In [3]:
# reopen the index without re-computing embeddings
from sentence_transformers import SentenceTransformer
from sqlitesearch import VectorSearchIndex

# load the model
model = SentenceTransformer("all-MiniLM-L6-v2")

# create the index
vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode="ivf",
    db_path="faq_vectors2.db"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [4]:
from rag_helper import RAGBase

# extend the base RAG class
class RAGVector(RAGBase):

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {"course": self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )

# create the assistant with sqlite search
vector_assistant = RAGVector(
    embedder=model,
    index=vs_index,
    llm_client=openai_client,
)

In [5]:
# try the sqlite search
vector_assistant.rag("the program has already begun, can I still sign up?")

'Yes, you can still join. If you want a certificate, make sure to submit your project while submissions are still open.'

In [6]:
# when done, close the connection
vs_index.close()